In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from torch.utils.data import TensorDataset, DataLoader

# -------------------------------------------------
# Paths
# -------------------------------------------------
# -------------------------------------------------
# Paths
# -------------------------------------------------
HERE = Path.cwd()

if (HERE / "data").exists() and (HERE / "results").exists():
    PROJECT_ROOT = HERE
elif (HERE.parent / "data").exists() and (HERE.parent / "results").exists():
    PROJECT_ROOT = HERE.parent
else:
    raise FileNotFoundError(
        f"Could not locate repo root from working directory: {HERE}"
    )

DATA_PATH = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "intermediate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = RESULTS_DIR / "bnn_table2-3.csv"


# -------------------------------------------------
# Terrain data
# -------------------------------------------------
terrain_path = DATA_PATH / "weightedTerrainData.csv"
terrain_df = pd.read_csv(terrain_path)

# Keep only numeric terrain columns except an obvious turbine-id column if present
terrain_num = terrain_df.select_dtypes(include=[np.number]).copy()

# If first numeric column is just turbine index/id, drop it.
# Adjust this if your file structure is different.
if terrain_num.shape[1] > 3:
    terrain_num = terrain_num.iloc[:, 1:]

# Scale terrain globally once
terrain_scaler = StandardScaler()
terrain_scaled = terrain_scaler.fit_transform(terrain_num.values.astype(np.float32))


# -------------------------------------------------
# Config
# -------------------------------------------------
TURBINES = list(range(1, 67))
TESTSET_2018 = list(range(1, 47)) + [48, 49, 50, 52] + list(range(54, 61)) + list(range(62, 67))
FEATURES = ["wind_speed", "temperature"]
TARGET = "power"

SEED = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# BNN settings: same style as your uploaded code
BATCH_SIZE = 4096
EPOCHS = 6
LR = 1e-3

H = 128
PRIOR_MU = 0.0
PRIOR_SIGMA = 0.1
KL_WEIGHT = 0.01
MC_DRAWS = 40

print("Device:", DEVICE)
# Map turbine id -> terrain vector
terrain_map = {
    tid: terrain_scaled[tid - 1]   # assumes row 1 = turbine 1, ..., row 66 = turbine 66
    for tid in TURBINES
}
def build_xy_with_terrain(df, tid, feature_cols, target_col):
    df = df[feature_cols + [target_col]].dropna().copy()

    X_dyn = df[feature_cols].to_numpy(np.float32)

    terr = terrain_map[tid].astype(np.float32)
    terr_mat = np.repeat(terr.reshape(1, -1), repeats=len(df), axis=0)

    X = np.column_stack([X_dyn, terr_mat]).astype(np.float32)
    y = df[target_col].to_numpy(np.float32)

    return X, y
# -------------------------------------------------
# Seed
# -------------------------------------------------
def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

# -------------------------------------------------
# Helper functions
# -------------------------------------------------
def read_csv_safe(path):
    try:
        path = Path(path)
        if path.exists():
            return pd.read_csv(path)
    except Exception:
        pass
    return None


def nlpd_gaussian(y, mu, sd, sd_floor=1e-6):
    sd = np.maximum(sd, sd_floor)
    return float(np.mean(
        0.5 * np.log(2.0 * np.pi * sd**2) +
        0.5 * ((y - mu) ** 2) / (sd ** 2)
    ))


# -------------------------------------------------
# Model: SAME structure as uploaded code style
# -------------------------------------------------
def make_model(d_in):
    model = nn.Sequential(
        bnn.BayesLinear(
            prior_mu=PRIOR_MU,
            prior_sigma=PRIOR_SIGMA,
            in_features=d_in,
            out_features=H
        ),
        nn.ReLU(),
        bnn.BayesLinear(
            prior_mu=PRIOR_MU,
            prior_sigma=PRIOR_SIGMA,
            in_features=H,
            out_features=H
        ),
        nn.ReLU(),
        bnn.BayesLinear(
            prior_mu=PRIOR_MU,
            prior_sigma=PRIOR_SIGMA,
            in_features=H,
            out_features=1
        ),
    ).to(DEVICE)
    return model


mse_loss = nn.MSELoss()
kl_loss = bnn.BKLLoss(reduction="mean", last_layer_only=False)


def train_bnn(model, X_train, y_train):
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        drop_last=False
    )

    optimizer = optim.Adam(model.parameters(), lr=LR)

    model.train()
    for epoch in range(EPOCHS):
        epoch_mse = 0.0
        epoch_kl = 0.0
        n_batches = 0

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            pred = model(xb)
            mse = mse_loss(pred, yb)
            kl = kl_loss(model)
            loss = mse + KL_WEIGHT * kl

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            epoch_mse += mse.item()
            epoch_kl += kl.item()
            n_batches += 1

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"MSE: {epoch_mse / max(n_batches, 1):.4f} | "
            f"KL: {epoch_kl / max(n_batches, 1):.4f}"
        )

    return model


@torch.no_grad()
def predict_mc(model, X_test, mc_draws=MC_DRAWS, pred_batch_size=8192):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    n = X_tensor.shape[0]
    draws = []

    for _ in range(mc_draws):
        preds = []
        for start in range(0, n, pred_batch_size):
            xb = X_tensor[start:start + pred_batch_size].to(DEVICE)
            pred = model(xb).cpu().numpy().reshape(-1)
            preds.append(pred)
        draws.append(np.concatenate(preds))

    draws = np.stack(draws, axis=0)   # (mc_draws, n_test)
    pred_mean = draws.mean(axis=0)
    pred_sd = draws.std(axis=0)

    return pred_mean, pred_sd


# -------------------------------------------------
# Main loop
# -------------------------------------------------
rows = []

for i in TURBINES:
    print(f"\n=== Evaluating Turbine {i} ===")

    test17 = read_csv_safe(DATA_PATH / f"Turbine{i}_2017.csv")
    test18 = read_csv_safe(DATA_PATH / f"Turbine{i}_2018.csv") if i in TESTSET_2018 else None

    if test17 is None or not set(FEATURES + [TARGET]).issubset(test17.columns):
        print(f"Skipping Turbine {i} (missing 2017 data or columns)")
        continue

    # ----------------------------
    # Training data: all other turbines in 2017
    # ----------------------------
    X_train_parts = []
    y_train_parts = []

    for j in TURBINES:
        if j == i:
            continue

        tr = read_csv_safe(DATA_PATH / f"Turbine{j}_2017.csv")
        if tr is None or not set(FEATURES + [TARGET]).issubset(tr.columns):
            continue

        Xj, yj = build_xy_with_terrain(tr, j, FEATURES, TARGET)
        if len(yj) == 0:
            continue

        X_train_parts.append(Xj)
        y_train_parts.append(yj)

    if len(X_train_parts) == 0:
        print(f"Skipping Turbine {i} (no training data)")
        continue

    X_train = np.vstack(X_train_parts).astype(np.float32)
    y_train = np.concatenate(y_train_parts).astype(np.float32)

    # ----------------------------
    # Scale X and y using train only
    # ----------------------------
    x_scaler = StandardScaler()
    X_train_s = X_train.copy()
    X_train_s[:, :len(FEATURES)] = x_scaler.fit_transform(X_train[:, :len(FEATURES)]).astype(np.float32)
    
    y_mean = float(np.mean(y_train))
    y_std = float(np.std(y_train))
    if not np.isfinite(y_std) or y_std < 1e-8:
        y_std = 1.0

    y_train_s = ((y_train - y_mean) / y_std).astype(np.float32)

    # ----------------------------
    # Train
    # ----------------------------
    t1 = time.time()
    model = make_model(d_in=X_train_s.shape[1])
    model = train_bnn(model, X_train_s, y_train_s)
    t2 = time.time()
    train_time = round(t2 - t1, 2)

    # ----------------------------
    # Test on 2017
    # ----------------------------

    X_te17, y_te17 = build_xy_with_terrain(test17, i, FEATURES, TARGET)
    
    X_te17_s = X_te17.copy()
    X_te17_s[:, :len(FEATURES)] = x_scaler.transform(X_te17[:, :len(FEATURES)]).astype(np.float32)
    
    p1 = time.time()
    pred17_s, pred17_sd_s = predict_mc(model, X_te17_s, mc_draws=MC_DRAWS)
    p2 = time.time()

    pred17 = pred17_s * y_std + y_mean
    pred17_sd = pred17_sd_s * y_std

    pred17 = np.clip(pred17, 0.0, None)
    pred17_time = round(p2 - p1, 2)

    rmse17 = float(np.sqrt(mean_squared_error(y_te17, pred17)))
    nlpd17 = nlpd_gaussian(y_te17, pred17, pred17_sd)

    # ----------------------------
    # Test on 2018
    # ----------------------------
    if test18 is not None and set(FEATURES + [TARGET]).issubset(test18.columns):
        test18 = test18[FEATURES + [TARGET]].dropna()

        if not test18.empty:

            X_te18, y_te18 = build_xy_with_terrain(test18, i, FEATURES, TARGET)

            X_te18_s = X_te18.copy()
            X_te18_s[:, :len(FEATURES)] = x_scaler.transform(X_te18[:, :len(FEATURES)]).astype(np.float32)
            
            p3 = time.time()
            pred18_s, pred18_sd_s = predict_mc(model, X_te18_s, mc_draws=MC_DRAWS)
            p4 = time.time()

            pred18 = pred18_s * y_std + y_mean
            pred18_sd = pred18_sd_s * y_std

            pred18 = np.clip(pred18, 0.0, None)
            pred18_time = round(p4 - p3, 2)

            rmse18 = float(np.sqrt(mean_squared_error(y_te18, pred18)))
            nlpd18 = nlpd_gaussian(y_te18, pred18, pred18_sd)
        else:
            rmse18 = np.nan
            nlpd18 = np.nan
            pred18_time = np.nan
    else:
        rmse18 = np.nan
        nlpd18 = np.nan
        pred18_time = np.nan

    row = {
        "Turbine": i,
        "Method": "BNN",
        "RMSE_2017": rmse17,
        "NLPD_2017": nlpd17,
        "RMSE_2018": rmse18,
        "NLPD_2018": nlpd18,
        "Runtime_train": train_time,
        "Runtime_pred17": pred17_time,
        "Runtime_pred18": pred18_time
    }
    rows.append(row)
    print(pd.DataFrame([row]))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# -------------------------------------------------
# Save intermediate results
# -------------------------------------------------
results = pd.DataFrame(
    rows,
    columns=[
        "Turbine", "Method",
        "RMSE_2017", "NLPD_2017",
        "RMSE_2018", "NLPD_2018",
        "Runtime_train", "Runtime_pred17", "Runtime_pred18"
    ]
)

results.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
print(results.head())

# -------------------------------------------------
# Update final results.csv
# -------------------------------------------------
sys.path.insert(0, str(PROJECT_ROOT / "code"))
from update_final_results import update_final_results

# Table 2 (2017)
table2_df = results.dropna(subset=["RMSE_2017"]).copy()
bnn_table2_rmse = table2_df["RMSE_2017"].mean() if not table2_df.empty else np.nan
bnn_table2_nlpd = table2_df["NLPD_2017"].mean() if not table2_df.empty else np.nan
bnn_table2_runtime = (
    (table2_df["Runtime_train"] + table2_df["Runtime_pred17"]).mean()
    if not table2_df.empty else np.nan
)

update_final_results(
    method="BNN",
    table_id="Table 2",
    rmse=bnn_table2_rmse,
    nlpd=bnn_table2_nlpd,
    runtime=bnn_table2_runtime
)

# Table 3 (2018)
table3_df = results.dropna(subset=["RMSE_2018"]).copy()
bnn_table3_rmse = table3_df["RMSE_2018"].mean() if not table3_df.empty else np.nan
bnn_table3_nlpd = table3_df["NLPD_2018"].mean() if not table3_df.empty else np.nan
bnn_table3_runtime = (
    (table3_df["Runtime_train"] + table3_df["Runtime_pred18"]).mean()
    if not table3_df.empty else np.nan
)

update_final_results(
    method="BNN",
    table_id="Table 3",
    rmse=bnn_table3_rmse,
    nlpd=bnn_table3_nlpd,
    runtime=bnn_table3_runtime
)

print("\nDone.")